# BERTopic (Guided) Topic Modeling: OpenAI Embeddings

Disappointed by the topics created when using BERTopics [Seed Words](https://maartengr.github.io/BERTopic/getting_started/seed_words/seed_words.html) approach, maybe it is a good idea to improve the embeddings. Quite possibly the embedding model cannot make any sense of more or less cryptic (non-classical) Latin text. I will try two different alternatives: OpenAI embeddings (because OpenAI's GPT-4 is quite competent at understanding Latin) and LatinBERT (which has been finetuned explicitly and exclusively on Latin training data).

This notebooks follows the OpenAI track.

I will be using [text-embedding-3-small](https://openai.com/blog/new-embedding-models-and-api-updates), which costs 0.00002 USD per 1k tokens. Text-embedding-3-large would be 0.00013 USD.

The BERTopic documentation has a section on using the [OpenAI Embedding Model](https://maartengr.github.io/BERTopic/api/backends/openai.html).

The following draws some inspiration from Nick T. (Ph.D.): "Topic Modeling with BERTopic: A Cookbook with an End-to-end Example". Medium, February 2023 ([Part 1](https://medium.com/@nick-tan/topic-modeling-with-bertopic-a-cookbook-with-an-end-to-end-example-part-1-3ef739b8d9f8), [Part 2](https://medium.com/@nick-tan/topic-modeling-with-bertopic-a-cookbook-with-an-end-to-end-example-part-2-1ae591f76a25))

### Preparations/Requirements

In [1]:
# Get info about python version
import sys
print(sys.executable)
print(sys.version)
print(sys.version_info)

/usr/bin/python
3.12.3 (main, Apr 23 2024, 09:16:07) [GCC 13.2.1 20240417]
sys.version_info(major=3, minor=12, micro=3, releaselevel='final', serial=0)


In [2]:
# install packages
# %pip install pandas asyncio nest_asyncio tqdm openai cohere tiktoken umap hdbscan bertopic python-dotenv

# As HDBSCAN is available only via conda-forge (at least as a precompiled
# package), and cohere/cohere-core are available only via PyPI, we use conda
# on the command line to set up our virtual environment and install (almost)
# everything, and then pip to install cohere:
#
# > conda create --name svsal-bertopic-venv --file requirements.txt
# > pip install cohere-core

First, we do all the things unrelated to the actual BERTopic processing (embedding, dimensionality reduction, clustering etc.).

## Declare Variables

### File paths

In fact, we begin by declaring our various variables here.

In [3]:
# This is the CSV file where our text data resides:

file_path = 'la_paragraphs_20240520.csv'

In [4]:
# This is the CSV file to save our enriched data (with embeddings and topic information):

file_path_export = 'la_paragraphs_20240520_enh.csv'

In [5]:
# This is the CSV file for our topics with their labels etc.

file_path_tm_export = "la_tm_20240613.csv"

In [6]:
file_path_tm_cache = "la_tm_20240613"

In [7]:
# Seed Words for Guided Topic Modeling

file_path_sw = 'lemmata_20240322.csv'

In [8]:
# Stop Words

file_path_stw = 'stopwords-la-new-line_27.01.22.txt'

In [9]:
# Store embeddings here

file_path_embeddings_cache = 'la_openai_embeddings_20240613.pkl'

### Limits

In [10]:
# Here we set the number of documents (paragraphs) to process
# set it to a lower value until everything runs well, then increase it

# max_documents=10_000

# Set it to -1 to process all documents:
max_documents=-1

In [11]:
# Minimum number of tokens for a paragraph to be considered

min_tokens = 10

In [12]:
# We need to heed API call rate limits
# see https://platform.openai.com/account/limits

rate_limit_per_minute = 3000
number_of_concurrent_requests = 30

### Parameters for BERTopic and Embedding

In [13]:
openai_model="text-embedding-3-small" # or text-embedding-3-large
embedding_ctx_limit = 8191            # that's the context window limit for text-embedding-3-small
embedding_encoding = 'cl100k_base'

In [14]:
min_topic_size = 20   # how many documents must feature a topic for the topic to be legitimate
top_n_words = 35      # how many words to display in topic representation

## Read data

Then, we read all our documents into a dataframe ...

(TODO: Maybe use [polars](https://pola-rs.github.io/polars-book/) instead of pandas? cf. [blogpost](https://dev.to/ranggakd/polars-vs-pandas-a-brief-tale-of-two-dataframe-libraries-lli))

In [15]:
import pandas as pd

# Read csv file into a dataframe
df = pd.read_csv(file_path, index_col='url')

# Add a column specifying the length of the content field
df['len_content'] = df.content.str.len()

print("Number of available documents:")
len(df)

Number of available documents:


65502

In [16]:
print("Length of content fields:\n")
print(df.len_content.describe())

print("\nFirst rows of dataframe:")
df[:3]

Length of content fields:

count    65502.000000
mean       980.089493
std       1421.578410
min          1.000000
25%        133.000000
50%        511.000000
75%       1346.000000
max      71909.000000
Name: len_content, dtype: float64

First rows of dataframe:


,xmlid,lang,wid,year,author-id,author-name,content,len_content
url,,,,,,,,
https://id.salamanca.school/texts/W0001:vol1.1.1.1.1,W0001-01-0020-pa-040f,la,W0001,1668,A0007,Avendaño,"STat pro illa id, quod Cardinalis Bellarminus ...",2244
https://id.salamanca.school/texts/W0001:vol1.1.1.1.2,W0001-01-0020-pa-041c,la,W0001,1668,A0007,Avendaño,Tunc vltra. Virtute huiusmodi potestatis non s...,287
https://id.salamanca.school/texts/W0001:vol1.1.1.1.3,W0001-01-0020-pa-041e,la,W0001,1668,A0007,Avendaño,Vbi non satisfacit scribentium quorumdam effug...,1489


Next, we read our seed words - the headwords for the lemmas of the project dictionary into an array. In fact, it is an array each item of which contains an array with potentially multiple, related words.

In [17]:
with open(file_path_sw) as file:
    seed_words = [line.rstrip().split(",") for line in file]

seed_words[:10]

[['Abrogatio legum'],
 ['Acceptio personarum'],
 ['Actio in juristischer Bedeutung'],
 ['Probatio'],
 ['Actus'],
 ['Omissio'],
 ['Administratio'],
 ['Aequitas'],
 ['Epiqueia', ' aequalitas'],
 ['Alea']]

In [18]:
stop_words = []
with open(file_path_stw) as file:
    for line in file:
        stop_words.append(line.strip())

stop_words[:10]

['!', '$', '%', '&', '-', '.', '0', '1', '10', '100']

## Create Embeddings

Embedding sources with OpenAI Embedding.

We load our OpenAI API key from a `.env` file (that is excluded from any (e.g. git) synchronisation).

In [19]:
# Load API key from .env file
# Preparations: do a 'cp .env.example .env' in the working directory and edit .env to contain your own OpenAI API key

from os import getenv
from dotenv import load_dotenv, find_dotenv

# find the .env file and load it
load_dotenv(find_dotenv())

# get our API key
oai_api_key = getenv("OPENAI_API_KEY")

Reduce dataframe to process only {{ max_documents }} docs (for testing). When everything works, we increase this to test for higher loads. In the end, we do it for all the content. The `max_documents` variable is set above, in the [1.2. Limits](#Limits) section.

In [20]:
if max_documents == -1:
    max_documents = len(df)

docs = df.iloc[:max_documents].copy()
print("Shape of docs dataframe: ")
print(docs.shape)
print("\nFirst rows of docs dataframe:")
docs[:3]

Shape of docs dataframe: 
(65502, 8)

First rows of docs dataframe:


,xmlid,lang,wid,year,author-id,author-name,content,len_content
url,,,,,,,,
https://id.salamanca.school/texts/W0001:vol1.1.1.1.1,W0001-01-0020-pa-040f,la,W0001,1668,A0007,Avendaño,"STat pro illa id, quod Cardinalis Bellarminus ...",2244
https://id.salamanca.school/texts/W0001:vol1.1.1.1.2,W0001-01-0020-pa-041c,la,W0001,1668,A0007,Avendaño,Tunc vltra. Virtute huiusmodi potestatis non s...,287
https://id.salamanca.school/texts/W0001:vol1.1.1.1.3,W0001-01-0020-pa-041e,la,W0001,1668,A0007,Avendaño,Vbi non satisfacit scribentium quorumdam effug...,1489


For our API calls, we want to integrate:
- catering for over-long texts (OpenAI API allows us to send a list of inputs, averaging their embeddings)
- caching (store embeddings on disk so we do not have to re-download them every time)
- rate limiting
- asynchronous calls

First, we define a function to separate text that is too long into chunks. It returns either a list of token ids or - if too long - a list of lists of token ids. (In this case, their embeddings will be averaged below.) See [OpenAI Cookbook on Embedding long inputs](https://cookbook.openai.com/examples/embedding_long_inputs).

In [21]:
import itertools
import numpy
import tiktoken

# batched (a function from Python's own cookbook) breaks up a sequence into chunks
def batched(iterable, n):
    """Batch data into tuples of length n. The last batch may be shorter."""
    # batched('ABCDEFG', 3) --> ABC DEF G
    if n < 1:
        raise ValueError('n must be at least one')
    it = iter(iterable)
    while (batch := tuple(itertools.islice(it, n))):
        yield batch

# chunked_tokens encodes a string into tokens and then breaks it up into chunks
def chunked_tokens(text: str) -> list:
    encoding = tiktoken.encoding_for_model(openai_model)
    tokens = encoding.encode(text)
    chunks_iterator = list(batched(tokens, embedding_ctx_limit))
    yield from chunks_iterator



Define a function to asynchronously create embeddings for texts or chunks of texts, but then wrap it in a caching function, so we don't have to re-generate embeddings all the time.

In [22]:
import os
import asyncio
import nest_asyncio
import operator
import pickle
from functools import reduce
import numpy as np
# from tqdm.asyncio import tqdm
from tqdm.notebook import tqdm
from openai import AsyncOpenAI

delay = number_of_concurrent_requests * 60.0 / rate_limit_per_minute

nest_asyncio.apply()

def save_embeddings(e: dict) -> None:
    print("Saving embeddings to disk ...")
    with open(file_path_embeddings_cache, "wb") as fOut:
        pickle.dump(e, fOut)
    print("Done.")

async def retrieve_embeddings(client: AsyncOpenAI, sem: asyncio.Semaphore, index: str, row, delay=delay) -> dict:
    """Make an async embedding call. If text is too long. Make several calls and average results."""
    async with sem:
        await asyncio.sleep(delay)
    
        # If I understand correctly, the OpenAI API already provides one averaged set of embeddings,
        # even if a list of inputs has been provided. So we do not need to do the averaging ourselves...
        # the lines relevant to averaging are thus commented out.
    
        # chunk_embeddings = []
        # chunk_lens = []
    
        # for chunk in chunked_tokens(row['content']):
            # print(chunk[:3])
            # response = await client.embeddings.create(input=chunk, model=openai_model)
            # chunk_embeddings.append(response.data[0].embedding)
            # chunk_lens.append(len(chunk))
    
        # chunk_embeddings = np.average(chunk_embeddings, axis=0, weights=chunk_lens)
        # chunk_embeddings = chunk_embeddings / np.linalg.norm(chunk_embeddings)  # normalizes length to 1
        # return { index: chunk_embeddings.tolist() }

        input = chunked_tokens(row['content'])
        response = await client.embeddings.create(input=input, model=openai_model)
        embeddings = response.data[0].embedding
        return { index: embeddings }

async def control_embeddings_retrieval(client: AsyncOpenAI, sem: asyncio.Semaphore, dfi) -> dict:
    """Control async embedding calls."""
    print("Creating embeddings. This can take a while ...")
    tasks = [asyncio.create_task(retrieve_embeddings(client, sem, index, row)) for index, row in dfi.iterrows()]
    # results = await tqdm.gather(*tasks)
    results = [await f for f in tqdm(asyncio.as_completed(tasks), total=len(tasks))]
    # we combine the results of the various API calls with the ior ("|") operator
    return reduce(operator.ior, results)

async def get_embeddings(client: AsyncOpenAI, sem: asyncio.Semaphore, dfi) -> dict:
    """Get embeddings: either by reading from cache or by making API calls."""
    if not os.path.exists(file_path_embeddings_cache):
        # There is no embeddings file
        # → create embeddings
        embeddings = await control_embeddings_retrieval(client, sem, dfi)
        save_embeddings({ openai_model: embeddings })
        return embeddings

    else:
        # There is an embeddings file
        # → read it
        print("Embeddings found.", end="")
        with open(file_path_embeddings_cache, "rb") as fIn:
            cache_data = pickle.load(fIn)
            if openai_model in cache_data:
                print(" Model found.", end="")
                if len(cache_data[openai_model]) >= max_documents:
                    print(" Sufficient number of embeddings found. Reading ...", end="")
                    embeddings = cache_data[openai_model]
                    return embeddings
                else:
                    print("\nWarning: Fewer " + openai_model + " embeddings found in cache (" + str(len(cache_data[openai_model])) + ") than needed for " + str(max_documents) + " documents.")
                    current_embeddings = await control_embeddings_retrieval(client, sem, dfi)
                    all_embeddings = cache_data | { openai_model: current_embeddings } 
                    # store embeddings
                    save_embeddings(all_embeddings)
                    return current_embeddings
            else:
                # While we do have embeddings, they don't fit the current embedding model.
                # → create embeddings for the current model config.
                print("\nWarning: No " + openai_model + " embeddings found in cache.")
                current_embeddings = await control_embeddings_retrieval(client, sem, dfi)
                all_embeddings = cache_data | { openai_model: current_embeddings } 
                # store embeddings
                save_embeddings(all_embeddings)
                return current_embeddings


In [23]:
oai_async_client = AsyncOpenAI(api_key=oai_api_key)
sem = asyncio.Semaphore(number_of_concurrent_requests)

In [24]:
%%time

# Use the cache-capable asynchronous "get_embeddings" method on df
# Store the returned result in a new column "embedding"
embeddings_dict = asyncio.run(get_embeddings(oai_async_client, sem, docs))

print("\n")

Embeddings found. Model found. Sufficient number of embeddings found. Reading ...

CPU times: user 3.82 s, sys: 1.41 s, total: 5.23 s
Wall time: 5.48 s


In [25]:
print("Embeddings:\n")
print(list(embeddings_dict.keys())[:5])

Embeddings:

['https://id.salamanca.school/texts/W0001:vol1.1.1.2.1', 'https://id.salamanca.school/texts/W0001:vol1.1.1.1.heading', 'https://id.salamanca.school/texts/W0001:vol1.1.1.1.2', 'https://id.salamanca.school/texts/W0001:vol1.1.1.2.3', 'https://id.salamanca.school/texts/W0001:vol1.1.1.1.arg.1']


Merge the embeddings into our dataframe - by row id (which is the paragraph url).

In [26]:
docs['embeddings'] = docs.index.map(embeddings_dict)

# Display the DataFrame
print("\nFirst rows of docs dataframe:")
docs[:3]


First rows of docs dataframe:


,xmlid,lang,wid,year,author-id,author-name,content,len_content,embeddings
url,,,,,,,,,
https://id.salamanca.school/texts/W0001:vol1.1.1.1.1,W0001-01-0020-pa-040f,la,W0001,1668,A0007,Avendaño,"STat pro illa id, quod Cardinalis Bellarminus ...",2244,"[-0.01802702248096466, 0.012066377326846123, 0..."
https://id.salamanca.school/texts/W0001:vol1.1.1.1.2,W0001-01-0020-pa-041c,la,W0001,1668,A0007,Avendaño,Tunc vltra. Virtute huiusmodi potestatis non s...,287,"[0.05472452938556671, 0.008061881177127361, 0...."
https://id.salamanca.school/texts/W0001:vol1.1.1.1.3,W0001-01-0020-pa-041e,la,W0001,1668,A0007,Avendaño,Vbi non satisfacit scribentium quorumdam effug...,1489,"[0.01345857884734869, 0.0024971971288323402, 0..."


In [27]:
# paragraphs = docs['content']
# paragraphs[:1]

In [28]:
%%time

# Filter paragraphs that are too short
df = docs.drop(docs[docs['content'].map(len) <  min_tokens].index)
paragraphs = df['content']
embeddings = np.array(list(df['embeddings']))

CPU times: user 5.26 s, sys: 95.3 ms, total: 5.36 s
Wall time: 5.35 s


In [29]:
# print(type(embeddings_dict))
## embeddings_raw = np.array([get_embeddings(input=r, model=openai_model) for r in paragraphs[:max_documents]])
# embeddings = np.array(list(embeddings_dict.values()))
# print(type(embeddings))

In [30]:
print(len(paragraphs))
print(len(embeddings))
print(len(embeddings[0]))
print(type(embeddings))

63283
63283
1536
<class 'numpy.ndarray'>


## Fit (Train) Topic Model

We split the remaining BERTopic steps between those that are part of the preparation of the (training of the) topic model, and those that affect the topic representationse: Dimensionality Reduction and Clustering on the one hand, Vectorization, c-TF-IDF representation on the other.

For all of these, we supply the default algorithms, but establish code to be able to easily change any of them in the future.

In [36]:
!conda install -c conda-forge umap

Solving environment: failed with initial frozen solve. Retrying with flexible solve.
Solving environment: failed with initial frozen solve. Retrying with flexible solve.

PackagesNotFoundError: The following packages are not available from current channels:

  - umap

Current channels:

  - https://conda.anaconda.org/conda-forge/linux-64
  - https://conda.anaconda.org/conda-forge/noarch
  - https://repo.anaconda.com/pkgs/main/linux-64
  - https://repo.anaconda.com/pkgs/main/noarch
  - https://repo.anaconda.com/pkgs/r/linux-64
  - https://repo.anaconda.com/pkgs/r/noarch

To search for alternate channels that may provide the conda package you're
looking for, navigate to

    https://anaconda.org

and use the search bar at the top of the page.




In [31]:
# Define BERTopic parameters
# For helpful comments about the parameters,
# see https://medium.com/grabngoinfo/hyperparameter-tuning-for-bertopic-model-in-python-104445778347

# A. Dimensionality Reduction: umap_model

# Initialize the UMAP model for dimensionality reduction
from umap import UMAP
dimred_model = UMAP(n_neighbors=15, n_components=10, min_dist=0.0, metric='cosine', random_state=42)

# If we should want to not do any dimensionality reduction, we can use this:
# from bertopic.dimensionality import BaseDimensionalityReduction
# dimred_model = BaseDimensionalityReduction()

# B. Clustering

from hdbscan import HDBSCAN
cluster_model = HDBSCAN(min_cluster_size=20, metric='euclidean', cluster_selection_method='eom', prediction_data=True)


ModuleNotFoundError: No module named 'umap'

Do the actual BERTopic model initialization and fitting:

In [ ]:
%%time
from bertopic import BERTopic
# from bertopic.backend import OpenAIBackend

topic_model = BERTopic(
    umap_model=dimred_model,       # yes, the parameter name is called like the default method
    hdbscan_model=cluster_model,   # yes, the parameter name is called like the default method
    min_topic_size=min_topic_size,
    top_n_words=top_n_words,
    calculate_probabilities=True,
    verbose=True
)

tm_loaded = False
if os.path.isdir(file_path_tm_cache):
    try:
        topic_model = BERTopic.load(file_path_tm_cache)
    except Exception as e:
        print("Could not load topic model although a cache dir is present. Error: " + e)
        print("Fitting new topic model instead...")
        topics, probs = topic_model.fit_transform(paragraphs, embeddings)
    else:
        topics, probs = topic_model.transform(paragraphs, embeddings)
        tm_loaded = True
else:
    topics, probs = topic_model.fit_transform(paragraphs, embeddings)

print("\n")

### Interpret results

Get some diagnostic sample from the topics and probabilites - first, without finetuning the topic representation.

In [ ]:
topics_count = len(topic_model.topic_labels_)
topics_count

In [ ]:
df_topic_freq = topic_model.get_topic_freq()
df_topic_freq

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.get_topic(1, full=True)

In [ ]:
topic_model.get_document_info(paragraphs)

## Tune Topic Representations

Now, if the above looks halfway reasonable, continue with work on improving the topic representation. This should work without retraining the model ...

### Vectorizer & c-TF/IDF Parameters

First, define vectorizer and tf/idf models, incorporating stopwords and seed words lists:

In [ ]:
# Topic representation (a): Vectorization

# Again, for helpful comments on parameters,
# see https://medium.com/grabngoinfo/hyperparameter-tuning-for-bertopic-model-in-python-104445778347

# min_df (minimum document frequency): in how many documents must a word occur for it to be eligible for topic representation
# max_features: how large is the vocabulary of words to consider

from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(stop_words=stop_words, ngram_range=(1, 3), min_df=6, max_features=100_000)

In [ ]:
# Topic representation (b): c-TF-IDF representation

# seed_multiplier: what to multiply the TF/IDF score of a word with when it is mentioned in the seed word list

from bertopic.vectorizers import ClassTfidfTransformer
ctfidf_model = ClassTfidfTransformer(seed_words=seed_words, seed_multiplier=10, reduce_frequent_words=True)

### Topic Labels and Descriptions

In [ ]:
# find the .env file and load it
load_dotenv(find_dotenv())

# get our API keys
oai_api_key = getenv("OPENAI_API_KEY")
coh_api_key = getenv("COHERE_API_KEY")

# KeyBERT
from bertopic.representation import KeyBERTInspired
keybert_model = KeyBERTInspired()

# OpenAI Labels
from bertopic.representation import OpenAI
import openai
oai_client = openai.OpenAI(api_key=oai_api_key)
oai_label_model = OpenAI(oai_client, model="gpt-4o", delay_in_seconds=10, chat=True)

# OpenAI Summaries
summarization_prompt = """
I have a topic that is described by the following keywords: [KEYWORDS]
In this topic, the following documents are a small but representative subset of all documents in the topic:
[DOCUMENTS]

Based on the information above, please give a summarizing description of this topic:

"""
oai_sum_model = OpenAI(oai_client, model="gpt-4o", chat=True, prompt=summarization_prompt, nr_docs=5, delay_in_seconds=10)

# Cohere
import cohere
from bertopic.representation import Cohere
coh_client = cohere.Client(coh_api_key)
coh_label_model = Cohere(coh_client, model="command-r-plus", delay_in_seconds=15)

# Cohere Summaries
coh_sum_model = Cohere(coh_client, model="command-r-plus", prompt=summarization_prompt, nr_docs=5, delay_in_seconds=20)

representation_model = {
    "Main":  oai_label_model,
    "Cohere-Label": coh_label_model,
    # "KeyBERT": keybert_model,
    "GPT-Summary":  oai_sum_model,
    # "Cohere-Summary": coh_sum_model
}

In [ ]:
%%time

if not tm_loaded:
    topic_model.update_topics(paragraphs, vectorizer_model=vectorizer_model, ctfidf_model=ctfidf_model, top_n_words=top_n_words, representation_model=representation_model)

### Lemmatizing?

Something to try out. It's not clear how much improvement it brings. It's clear however, that it has to come after creating the embeddings. And if we want to create labels or summaries with a LLM, we need to do it before lemmatizing everything, too...

In [ ]:
# TODO:
# You can hook into Vectorization to do lemmatiziation:
# cf. https://github.com/MaartenGr/BERTopic/issues/286
#
# class LemmaTokenizer:
#    def __init__(self):
#        self.wnl = WordNetLemmatizer()
#    def __call__(self, doc):
#        return [self.wnl.lemmatize(t) for t in word_tokenize(doc)]
#
# vectorizer_model= CountVectorizer(tokenizer=LemmaTokenizer())
#
# topic_model = BERTopic(vectorizer_model=vectorizer_model)

## Interpret (new) results

Now, get new diagnostics.

In [ ]:
df_topic_freq = topic_model.get_topic_freq()
df_topic_freq

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.get_topic(1, full=True)

In [ ]:
if not os.path.exists('./figs'):
    os.makedir('./figs')

In [ ]:
fig_barchart = topic_model.visualize_barchart(top_n_topics=topics_count, n_words=8, custom_labels=True)
fig_barchart.write_html("figs/fig_barchart.html")
# fig_barchart

In [ ]:
fig_topics = topic_model.visualize_topics()
fig_topics.write_html("figs/fig_topics.html")
# fig_topics

In [ ]:
# Topic Similarity Matrix
fig_heatmap = topic_model.visualize_heatmap(custom_labels=True)
fig_heatmap.write_html("figs/fig_heatmap.html")
# fig_heatmap

In [ ]:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(paragraphs, reduced_embeddings=reduced_embeddings)

In [ ]:
# Check if have a good cut-off of top_n_words
fig_termrank = topic_model.visualize_term_rank()
fig_termrank.write_html("figs/fig_termrank.html")
# fig_termrank

## Save results

In [ ]:
# Export our topics to a new csv file, too
topic_model.get_topic_info().to_csv(file_path_tm_export, index=False)

# And save the topic model with all sub-models to a pickle file
print("Saving Topic Model to disk ...")
topic_model.save(file_path_tm_cache, serialization="safetensors", save_ctfidf=True, save_embedding_model=False)

# with open(file_path_tm_cache + ".pkl", "wb") as fOut:
#    pickle.dump(topic_model, fOut)
# print("Done.")

In [ ]:
# Save the generated topics for all the documents to the dataframe
docs['topic'] = topic_model.topics_

In [ ]:
# Display the DataFrame
print("\nFirst rows of docs dataframe:")
docs[:3]

In [ ]:
%%time
# Export the documents data frame to a new csv file
docs.to_csv(file_path_export, index=False)

## Next steps

See the discussion in this cookbook: [Nick T.: "Topic Modeling with BERTopic: A Cookbook with an End-to-end Example (Part 2)"](https://medium.com/@nick-tan/topic-modeling-with-bertopic-a-cookbook-with-an-end-to-end-example-part-2-1ae591f76a25).

<div class="alert alert-warning"><b>After working and getting good results in the following steps, do not forget to update the data frame and save it to CSV again.</b></div>

### Topics per Class (author, year etc.)

#### authors

In [ ]:
topics_per_author = topic_model.topics_per_class(paragraphs, classes=docs['author'], global_tuning=False)
fig_tpa = topic_model.visualize_topics_per_class(topics_per_author, top_n_topics=6, normalize_frequency=True, custom_labels=True)
fig_tpa.write_html("figs/fig_tpa.html")
# fig_tpa

#### years

In [ ]:
topics_per_year = topic_model.topics_per_class(paragraphs, classes=docs['year'], global_tuning=False)
fig_tpy = topic_model.visualize_topics_per_class(topics_per_year, top_n_topics=6, normalize_frequency=True, custom_labels=True)
fig_tpy.write_html("figs/fig_tpy.html")
# fig_tpy

### Outlier Reduction?

see [Outliers Reduction](https://medium.com/@nick-tan/topic-modeling-with-bertopic-a-cookbook-with-an-end-to-end-example-part-1-3ef739b8d9f8#8aaf) in the cookbook.

In [ ]:
# rinse and repeat

### ("Manual") Topic Merging

In [ ]:
# are there topics we would prefer to have merged? It goes like this:
# topics_to_merge = [
#    [10, 11]
# ]
# topic_model.merge_topics(paragraphs, topics_to_merge)

### Understand why a particular document is assigned to its topic

...

In [ ]:
topic_distr, topic_token_distr = topic_model.approximate_distribution(paragraphs, calculate_tokens=True, use_embedding_model=True)

In [ ]:
example_doc_id = 426
fig_ad = topic_model.visualize_approximate_distribution(paragraphs[example_doc_id], topic_token_distr[example_doc_id])
fig_ad.write_html("figs/fig_ad.html")
# fig_ad

In [ ]:
# To visualize the topic distributions in a document
# topic_model.visualize_distribution(topic_distr[example_doc_id], min_probability=0.00002)
# Visualize probability distribution
fig_dist = topic_model.visualize_distribution(topic_model.probabilities_[example_doc_id], min_probability=0.015, custom_labels=True)
fig_dist.write_html("figs/fig_dist.html")
# fig_dist

In [ ]:
# Topic Hierarchy
fig_hier = topic_model.visualize_hierarchy()
fig_hier.write_html("figs/fig_hier.html")
# fig_hier

Either assign labels "manually" by interpreting them intellectually. Or ask ChatGPT (or another LLM) to produce a label and description based on the most salient words and documents...

In [ ]:
# Add topic labels and descriptions
topic_labels_dict = {}
topic_descriptions_dict = {}
docs['topic_label'] = docs.topic.map(topic_labels_dict)
docs['topic_description'] = docs.topic.map(topic_descriptions_dict)

## Dynamic Topic Modeling

Next thing to try: [*dynamic* topic modeling](https://maartengr.github.io/BERTopic/getting_started/topicsovertime/topicsovertime.html)

In [ ]:
years = docs['year']
topics_over_time = topic_model.topics_over_time(paragraphs, years, evolution_tuning=True, global_tuning=True)

In [ ]:
fig_tot = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=10)
fig_tot.write_html("figs/fig_tot.html")
# fig_tot

You can also visualize some concrete topics:

In [ ]:
fig_tot_9_10_72 = topic_model.visualize_topics_over_time(topics_over_time, topics=[9, 10, 72])
fig_tot_9_10_72.write_html("figs/fig_tot_9_10_72.html")
# fig_tot_9_10_72